In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

with rasterio.open('../data/raw/S2_Season1_BOA_Composite.tif') as src:
    data = src.read().astype(np.float32)

data = np.nan_to_num(data, nan=0.0)
B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12 = [data[i] for i in range(10)]

def safe_div(a, b):
    return np.where((a+b)!=0, (a-b)/(a+b), 0)

NDVI = safe_div(B8, B4)
EVI  = np.clip(2.5*np.where((B8+6*B4-7.5*B2+1)!=0,(B8-B4)/(B8+6*B4-7.5*B2+1),0),-1,1)
NDWI = safe_div(B3, B8)

# Rule-based classification
def classify(ndvi, ndwi, b11):
    c = np.zeros(ndvi.shape, dtype=np.uint8)
    c[ndvi > 0.4] = 1
    c[(ndvi > 0.2) & (ndvi <= 0.4)] = 2
    c[ndwi > 0.0] = 3
    c[(ndvi < 0.1) & (b11 > 0.2)] = 4
    c[(ndvi < 0.1) & (b11 <= 0.2)] = 5
    return c

classical_map = classify(NDVI, NDWI, B11)
print("Classical classification done")
print(f"Image shape: {data.shape}")

In [ ]:
h, w = B4.shape
features = np.stack([B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12,NDVI,EVI,NDWI], axis=-1)
features_flat = features.reshape(-1, 13)
labels_flat = classical_map.flatten()

np.random.seed(42)
valid_idx = np.where(labels_flat > 0)[0]
sample_idx = np.random.choice(valid_idx, min(10000, len(valid_idx)), replace=False)

X = features_flat[sample_idx]
y = labels_flat[sample_idx]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_scaled, y)

cv_scores = cross_val_score(rf, X_scaled, y, cv=3, scoring='accuracy')
print(f"RF CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

rf_map = rf.predict(scaler.transform(features_flat)).reshape(h, w)
print("Random Forest classification done")

In [ ]:
colors = ['black','darkgreen','yellowgreen','blue','sienna','gray']
labels = ['Unclassified','Dense Veg','Sparse Veg','Water','Bare Soil','Built-up']
cmap = mcolors.ListedColormap(colors)

rgb = np.stack([data[2], data[1], data[0]], axis=-1)
rgb = np.clip(rgb / 0.3, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb)
axes[0].set_title('True Colour RGB')
axes[0].axis('off')

im1 = axes[1].imshow(classical_map, cmap=cmap, vmin=0, vmax=5)
axes[1].set_title('Classical Rule-Based')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], ticks=range(6)).set_ticklabels(labels)

im2 = axes[2].imshow(rf_map, cmap=cmap, vmin=0, vmax=5)
axes[2].set_title(f'Random Forest\nCV Acc={cv_scores.mean():.3f}')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], ticks=range(6)).set_ticklabels(labels)

plt.suptitle('Task 4: Land-Cover Classification — Sahiwal, Punjab')
plt.tight_layout()
plt.savefig('../data/outputs/task4_notebook_output.png', dpi=150)
plt.show()